# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing the [FAIR^2 dataset package](https://doi.org/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All record sets, fields, and data elements are referenced by their `@id`, in line with the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print basic metadata from the Croissant Dataset object
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Explore the available record sets and their fields using their unique `@id` as defined by the Croissant metadata.

In [ ]:
# List available record sets with their @id and fields
print("Record sets and their fields, by @id:")
record_sets = dataset.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    for field in rs.fields:
        print(f"    - Field @id: {field.id} (name: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all dataframes for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(records_iter)
    dataframes[record_set_id] = df

# Print available DataFrames and their columns
first_rs = record_set_ids[0] if record_set_ids else None
if first_rs:
    print(f"Columns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing, such as filtering, normalization, and group-by aggregation. For demonstration, we select a numeric field by its `@id` from the previous overview (e.g., coverage, peptide counts, molecular weight, etc.), filter, normalize, and group if possible.

In [ ]:
# Pick a record set and field with numeric data.

# Select the main protein abundance/quantification record set (choose the first for example)
selected_record_set_id = first_rs
df = dataframes[selected_record_set_id]

# Attempt to pick a numeric field by heuristic (e.g., contains 'coverage', 'count', 'mw', or 'abundance')
numeric_candidates = [col for col in df.columns if any(k in col.lower() for k in ['coverage', 'count', 'mw', 'abundance', 'number', 'peptide'])]
print(f"Numeric field candidates: {numeric_candidates}")

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    # Coerce to numeric (errors='coerce' so we get NaN for non-numeric values)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].quantile(0.9)  # Example: filter top 10%
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt group by another field, e.g. 'description', 'accession', or other string columns
    groupable_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    if groupable_fields:
        group_field_id = groupable_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric fields found for demonstration.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. For demonstration, we create a histogram of one selected numeric field and a bar plot by group if possible.

In [ ]:
import matplotlib.pyplot as plt
# If a numeric field and filtered data exist, plot
if 'filtered_df' in locals() and numeric_candidates:
    plt.figure(figsize=(8, 4))
    filtered_df[numeric_field_id].hist(bins=15)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Optional: Bar plot for top groups
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(15)
        plt.bar(top_groups[group_field_id], top_groups[numeric_field_id])
        plt.title(f"Top 15 {group_field_id} by mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No visualization available due to missing numeric fields or empty data.")

## 6. Conclusion

In this notebook, you explored the FAIR^2 dataset describing mass spectrometry analysis of extracellular vesicle proteins from human mast cells. Using the `mlcroissant` library, you loaded the Croissant schema, examined the structure and metadata, loaded tabular records, and performed basic filtering, normalization, aggregation, and visualization using field and record set `@id` values. For advanced analysis, you can further explore the fields and customize data processing steps based on the rich Croissant metadata model.